# 07 - Professional Backtesting

Compare walk-forward, expanding, and rolling-window validation.


## Why Random Splits Are Dangerous

**Definition**  
Random splitting leaks future information into training folds for time series.

**Theory**  
Backtesting preserves chronology and simulates real deployment.

**Mathematical Intuition**  
Train on past `t<=T`; evaluate on future `t>T` repeatedly.

**Financial Intuition**  
Market regimes evolve; robust models should stay competitive across folds.

**Business Impact**  
Backtesting prevents false confidence from unrealistic validation.

**Real-World Example**  
A model that wins random split but fails walk-forward is not production-ready.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

from src.models import MODEL_REGISTRY

framework = make_framework()
framework.load_data()

rows = []
for strategy in ['walk_forward', 'expanding', 'rolling']:
    result = framework.backtest(horizon=1, model=MODEL_REGISTRY['Random Forest'], strategy=strategy)
    metrics = result['aggregated_metrics']
    rows.append({
        'strategy': strategy,
        'mean_rmse': metrics['mean_rmse'],
        'std_rmse': metrics['std_rmse'],
        'mean_mape': metrics['mean_mape'],
        'std_mape': metrics['std_mape'],
    })

bt_df = pd.DataFrame(rows).sort_values('mean_rmse')
display(bt_df)
bt_df.to_csv(OUT / 'tables/07_backtesting_h1.csv', index=False)
